# API Baseline: Dynamsoft Detector Results

This notebook is the external baseline for Dynamsoft. It loads the JSON returned by the Dynamsoft API for each scanned page, draws the returned location polygons, and builds a visual grid for review.


In [ ]:
from pathlib import Path
import json
import subprocess
import shutil
import math

from PIL import Image, ImageDraw

try:
    from IPython.display import display
except ImportError:
    def display(image):
        return image

def _find_root():
     for parent in [Path.cwd()] + list(Path.cwd().parents):
         if (parent / 'data').is_dir():
             return parent
     return Path.cwd()

ROOT = _find_root()
PDF_PATH = ROOT / 'data' / 'pdfs' / 'private-evaluation-document.pdf'

PAGES = list(range(1, 22))  # pages 1 .. 21
OUT_DIR = ROOT / 'output' / 'api-dynamsoft'
PAGE_DIR = OUT_DIR / 'rendered-pages'
DPI = 300
PG_JSON_DIR = ROOT / 'data' / 'jsons' / 'dynamsoft'
GRID_COLS = 2
CELL_WIDTH = 600

def find_pdftoppm():
    found = shutil.which('pdftoppm')
    if found:
        return found
    for candidate in [Path('/opt/homebrew/bin/pdftoppm'), Path('/usr/local/bin/pdftoppm')]:
        if candidate.exists():
            return str(candidate)
    cache_root = Path.home() / '.cache' / 'codex-runtimes'
    if cache_root.exists():
        for candidate in cache_root.glob('**/pdftoppm'):
            if candidate.is_file():
                return str(candidate)
    raise RuntimeError('pdftoppm not found. Install Poppler or add pdftoppm to PATH.')


def render_pdf_if_needed():
    PAGE_DIR.mkdir(parents=True, exist_ok=True)
    marker = PAGE_DIR / '.dpi'
    pages_ready = all((PAGE_DIR / f'page-{page:02d}.png').exists() for page in PAGES)
    dpi_ready = marker.exists() and marker.read_text().strip() == str(DPI)
    if pages_ready and dpi_ready:
        return
    for old_page in PAGE_DIR.glob('page-*.png'):
        old_page.unlink()
    subprocess.run(
        [find_pdftoppm(), '-r', str(DPI), '-png', str(PDF_PATH), str(PAGE_DIR / 'page')],
        check=True,
    )
    marker.write_text(str(DPI), encoding='utf-8')



Normalize the Dynamsoft API response and draw each returned barcode location on top of its source page.


In [ ]:
def load_detections(json_path):
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    if isinstance(data, list):
        return data

    if isinstance(data, dict):
        if isinstance(data.get('detections'), list):
            return data['detections']

        pages = data.get('pages')
        if isinstance(pages, list) and pages:
            detections = []
            for page in pages:
                detections.extend(page.get('detections', []))
            return detections

    raise ValueError('Unsupported JSON format: expected a list, detections, or pages[].detections')


def detection_points(det):
    loc = det.get('location')
    if isinstance(loc, list) and len(loc) >= 4:
        points = []
        for s in loc:
            x_str, y_str = s.split(',')
            points.append((int(x_str.strip()), int(y_str.strip())))
        return points

    if det.get('cornerPoints'):
        return [(p['x'], p['y']) for p in det['cornerPoints']]

    box = det.get('boundingBox') or det.get('rect')
    if not box:
        return None

    if all(k in box for k in ('left', 'top', 'right', 'bottom')):
        left, top, right, bottom = box['left'], box['top'], box['right'], box['bottom']
    elif all(k in box for k in ('x', 'y', 'width', 'height')):
        left, top = box['x'], box['y']
        right, bottom = left + box['width'], top + box['height']
    elif all(k in box for k in ('x1', 'y1', 'x2', 'y2')):
        left, top, right, bottom = box['x1'], box['y1'], box['x2'], box['y2']
    else:
        return None

    return [(left, top), (right, top), (right, bottom), (left, bottom)]


def detection_label(index, det):
    fmt = det.get('formatString') or det.get('format') or det.get('class_name') or det.get('label') or 'barcode'
    value = det.get('text') or det.get('rawValue') or det.get('value') or ''
    confidence = det.get('confidence')

    parts = [str(index), str(fmt)]
    if confidence is not None:
        parts.append(f'{float(confidence):.2f}')
    if value:
        parts.append(str(value))
    return ' '.join(parts)


def coordinate_scale(points, image_size):
    img_w, img_h = image_size
    flat = [p for pts in points for p in pts]
    if not flat:
        return 1.0, 1.0

    max_x = max(p[0] for p in flat)
    max_y = max(p[1] for p in flat)
    if max_x <= img_w and max_y <= img_h:
        return 1.0, 1.0

    for scale in (1 / 3, 1 / 2, 1 / 4):
        if max_x * scale <= img_w and max_y * scale <= img_h:
            return scale, scale

    return img_w / max_x, img_h / max_y


def draw_detections(image_path, json_path, output_path=None, line_width=3, draw_labels=False):
    image = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(image)
    detections = load_detections(json_path)

    colors = ['red', 'lime', 'blue', 'orange', 'magenta', 'cyan', 'yellow']

    img_w, img_h = image.size
    all_points = []
    valid_dets = []
    for index, det in enumerate(detections, start=1):
        pts = detection_points(det)
        if not pts:
            print(f'Skipped detection {index} in {Path(json_path).name}: no supported box fields')
            continue
        pts = [(float(x), float(y)) for x, y in pts]
        all_points.append(pts)
        valid_dets.append((index, det))

    scale_x, scale_y = coordinate_scale(all_points, image.size)
    if (scale_x, scale_y) != (1.0, 1.0):
        print(f'Scaling coordinates by x={scale_x:.4f}, y={scale_y:.4f} to fit image ({img_w}x{img_h})')

    for (index, det), pts in zip(valid_dets, all_points):
        points = [(p[0] * scale_x, p[1] * scale_y) for p in pts]
        color = colors[(index - 1) % len(colors)]
        draw.line(points + [points[0]], fill=color, width=line_width)

        if not draw_labels:
            continue

        label = detection_label(index, det)
        x = min(p[0] for p in points)
        y = min(p[1] for p in points)
        text_box = draw.textbbox((x, y), label)
        text_w = text_box[2] - text_box[0]
        text_h = text_box[3] - text_box[1]
        label_y = max(0, y - text_h - 6)
        draw.rectangle([x, label_y, x + text_w + 8, label_y + text_h + 6], fill=color)
        draw.text((x + 4, label_y + 3), label, fill='black')

    if output_path:
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        image.save(output_path)
        print(f'Saved: {output_path.resolve()}')

    return image


def create_grid(images, cols=2, cell_width=600):
    if not images:
        return None

    resized = []
    for img in images:
        ratio = cell_width / img.width
        new_h = int(img.height * ratio)
        resized.append(img.resize((cell_width, new_h), Image.LANCZOS))

    n = len(resized)
    rows_count = math.ceil(n / cols)
    row_heights = []
    for r in range(rows_count):
        start = r * cols
        end = min(start + cols, n)
        row_heights.append(max(resized[i].height for i in range(start, end)))

    total_h = sum(row_heights)
    grid = Image.new('RGB', (cell_width * cols, total_h))
    y = 0
    for r in range(rows_count):
        start = r * cols
        end = min(start + cols, n)
        for idx in range(start, end):
            img = resized[idx]
            x = (idx % cols) * cell_width
            y_offset = (row_heights[r] - img.height) // 2
            grid.paste(img, (x, y + y_offset))
        y += row_heights[r]
    return grid


In [ ]:
render_pdf_if_needed()
processed = []
for p in PAGES:
    img_path = PAGE_DIR / f'page-{p:02d}.png'
    json_path = PG_JSON_DIR / f'page-{p:02d}.json'
    if not json_path.exists():
        alt_path = PG_JSON_DIR / f'--page-{p:02d}.json'
        if alt_path.exists():
            json_path = alt_path
    if not img_path.exists():
        print(f'Skipping page {p}: image not found')
        continue
    if not json_path.exists():
        print(f'Skipping page {p}: JSON not found')
        continue
    print(f'Processing page {p}...')
    img = draw_detections(img_path, json_path)
    processed.append(img)

if processed:
    grid = create_grid(processed, cols=GRID_COLS, cell_width=CELL_WIDTH)
    print(f'Grid: {grid.size[0]}x{grid.size[1]}')
    display(grid)
else:
    print('No pages processed.')

